# World Cup & International Football — Exploratory Data Analysis

International men's football results from **1872 to 2026** (~49k matches), enriched
with goal-level events, Elo ratings and per-edition World Cup data.

**Goal:** understand the data, surface its quirks, and extract the historical
patterns that feed a *2026 World Cup forecasting* model (see [`experiments/`](../experiments)).

All loading/cleaning lives in [`src/worldcup/data.py`](../src/worldcup/data.py); every
figure and LaTeX table is produced through the NeurIPS-styled
[`worldcup.viz`](../src/worldcup/viz) layer — the notebook stays a narrative.

In [1]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
from worldcup import data, viz
from worldcup.viz import plots
viz.set_style()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Match results
One row per international fixture.

In [2]:
results = data.load_results()
played = data.load_results(played_only=True)
print(f"{len(results):,} fixtures | {results.date.min().date()} -> {results.date.max().date()}")
print(f"played: {len(played):,} | future/unplayed (NaN score): {len(results) - len(played)}")
results.head()

49,477 fixtures | 1872-11-30 -> 2026-06-27
played: 49,413 | future/unplayed (NaN score): 64


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.00,0.00,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.00,2.00,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.00,1.00,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.00,2.00,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.00,0.00,Friendly,Glasgow,Scotland,False


In [3]:
display(results.tournament.value_counts().head(8).to_frame("matches"))
print(f"goals/match: mean {played.total_goals.mean():.2f} | "
      f"median {played.total_goals.median():.0f} | max {played.total_goals.max()}")

,matches
tournament,
Friendly,18388
FIFA World Cup qualification,8771
UEFA Euro qualification,2824
African Cup of Nations qualification,2327
FIFA World Cup,1036
Copa América,869
African Cup of Nations,845
AFC Asian Cup qualification,829


goals/match: mean 2.94 | median 3 | max 31


In [4]:
viz.save_fig(plots.matches_per_year(played), "01_matches_per_year")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/01_matches_per_year.pdf')

**Goals per match** collapsed from 5–9 in the 1870s–1900s and has been flat at
~2.7 since the 1960s — football professionalized and defenses tightened.

In [5]:
viz.save_fig(plots.goals_per_match_trend(played), "02_goals_per_match")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/02_goals_per_match.pdf')

In [6]:
viz.save_fig(plots.goals_distribution(played), "03_goals_distribution")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/03_goals_distribution.pdf')

## 2. Home advantage
Restricting to **non-neutral** venues isolates the effect.

In [7]:
nz = played[~played.neutral]
print(f"home win {100*(nz.home_score>nz.away_score).mean():.1f}%  | "
      f"draw {100*(nz.home_score==nz.away_score).mean():.1f}%  | "
      f"away win {100*(nz.home_score<nz.away_score).mean():.1f}%")

home win 50.7%  | draw 22.9%  | away win 26.4%


In [8]:
viz.save_fig(plots.home_advantage_by_decade(played), "04_home_advantage")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/04_home_advantage.pdf')

Home advantage (~51% home wins) is stable across 120 years; the share of
**draws** is what crept up over time.

## 3. Team performance
Per-team match log → historical points percentage. Exported as a NeurIPS table.

In [9]:
log = data.team_match_log(played)
agg = (log.groupby("team")
          .agg(matches=("team","size"), wins=("win","sum"), draws=("draw","sum")))
agg["points_pct"] = (agg.wins*3 + agg.draws) / (agg.matches*3) * 100
best = (agg[agg.matches >= 200].sort_values("points_pct", ascending=False)
        .head(10).reset_index()[["team","matches","wins","points_pct"]])
best

,team,matches,wins,points_pct
0,Brazil,1060,672,70.22
1,Jersey,235,153,67.94
2,Spain,783,461,66.62
3,England,1090,625,65.23
4,Germany,1031,599,64.99
5,Iran,612,349,64.87
6,Guernsey,240,145,63.89
7,Argentina,1069,592,63.39
8,Italy,893,477,62.49
9,South Korea,1008,539,61.81


In [10]:
tex = viz.df_to_neurips_latex(
    best, label="tab:winrate",
    caption=("Historical points percentage of the top national teams "
             "(minimum 200 matches played; higher is better). "
             "Brazil leads despite the largest match load."),
    float_format="%.1f", bold_best="points_pct", lower_is_better=False)
print(viz.save_table(tex, "winrate"))

C:\Users\luisg\worldcup\reports\tables\winrate.tex


## 4. Goal events
Scorers, penalties and goal timing.

In [11]:
goals = data.load_goalscorers()
print(f"{len(goals):,} goals | penalties {100*goals.penalty.mean():.1f}% | "
      f"own goals {100*goals.own_goal.mean():.1f}%")
display(goals[~goals.own_goal].scorer.value_counts().head(10).to_frame("goals"))

47,620 goals | penalties 6.8% | own goals 1.9%


,goals
scorer,
Cristiano Ronaldo,121
Robert Lewandowski,69
Harry Kane,69
Romelu Lukaku,64
Lionel Messi,63
Edin Džeko,58
Aleksandar Mitrović,52
Luis Suárez,51
Ali Daei,49


In [12]:
viz.save_fig(plots.goal_minute_distribution(goals), "05_goal_minute")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/05_goal_minute.pdf')

## 5. Elo ratings
`load_elo` fixes the mixed ISO/US date encoding; `latest_elo` drops dissolved
nations (e.g. *West Germany*) from the current ranking.

In [13]:
latest = data.latest_elo(exclude_dissolved=True)
viz.save_fig(plots.top_elo(latest), "07_top_elo")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/07_top_elo.pdf')

In [14]:
elo = data.load_elo()
viz.save_fig(plots.elo_history(elo, ["Brazil","Spain","Germany","Argentina"]),
             "08_elo_history")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/08_elo_history.pdf')

## 6. World Cup editions
Goals per match per tournament, 1930–2026.

In [15]:
editions = data.load_worldcup_editions()
viz.save_fig(plots.worldcup_goals_per_match(editions), "06_worldcup_goals")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/06_worldcup_goals.pdf')

In [16]:
tex = viz.df_to_neurips_latex(
    editions, label="tab:wcgoals",
    caption=("Goals per match for every World Cup edition (1930–2026). "
             "Scoring peaked in 1954 and has hovered near 2.6 since the 1990s."),
    float_format="%.2f")
print(viz.save_table(tex, "worldcup_editions"))
editions

C:\Users\luisg\worldcup\reports\tables\worldcup_editions.tex


,year,matches,goals,goals_per_match
0,1930,18,70,3.89
1,1934,17,66,3.88
2,1938,18,75,4.17
3,1950,22,88,4.00
4,1954,26,136,5.23
5,1958,35,125,3.57
6,1962,32,89,2.78
7,1966,32,87,2.72
8,1970,32,88,2.75
9,1974,38,97,2.55


## Key takeaways
1. **Scoring** fell sharply early on, stable at ~2.7 goals/match since the 1960s.
2. **Home advantage** is durable (~51% home wins); draws rose over time.
3. **Brazil** leads historical points% among heavily-tested sides; **Spain** tops current Elo.
4. World Cup scoring peaked in **1954 (5.2)**, bottomed in **1990 (2.1)**, ~2.6 today.

### Data-quality notes (handled in `src/worldcup/data.py`)
- Elo dates come in two formats — single-format parsing nulls ~99% of rows.
- `results` includes future fixtures (NaN scores) — filtered via `played_only`.
- Dissolved nations coexist with current ones — join with `former_names` before aggregating.

**Next:** feature engineering + an Elo–Poisson hybrid to forecast the 2026 World Cup → [`experiments/`](../experiments).